# House Price Prediction Czech Republic

## 1. Setup & Data Loading

In [1]:
# DELETE columns containing sensitive information (personal identifiers like phone numbers)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score
import plotly.express as px

df_raw = pd.read_csv('images_and_data/sreality_master.csv')
print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

Dataset shape: (17757, 31)


,estate_id,title,locality,region,latitude,longitude,price_czk,price_per_m2,area_m2,category,...,vlastnictvi,energ_stitek,anuita,dist_vecerka_m,dist_obchod_m,dist_tram_m,dist_metro_m,dist_bus_mhd_m,url,scraped_at
0,1458614348,Prodej bytu 1+kk 36 m²,"Černá v Pošumaví, Český Krumlov",Jihočeský kraj,48.737864,14.102581,3290000.0,91389,36,1+kk,...,Osobní,C - Úsporná,NaN,723.0,NaN,NaN,NaN,481.0,https://www.sreality.cz/detail/prodej/byt/1+kk...,2026-07-18T21:41:29.040753+00:00
1,3749109836,Prodej bytu 1+kk 25 m²,"Litvínov - Horní Litvínov, Most",Ústecký kraj,50.610939,13.636989,899000.0,35960,25,1+kk,...,Družstevní,G - Mimořádně nehospodárná,NaN,28.0,1908.0,715.0,NaN,68.0,https://www.sreality.cz/detail/prodej/byt/1+kk...,2026-07-18T21:41:29.040753+00:00
2,569421900,Prodej bytu 1+kk 28 m²,"Černá v Pošumaví, Český Krumlov",Jihočeský kraj,48.737864,14.102581,2690000.0,96071,28,1+kk,...,Osobní,C - Úsporná,NaN,723.0,NaN,NaN,NaN,481.0,https://www.sreality.cz/detail/prodej/byt/1+kk...,2026-07-18T21:41:29.040753+00:00
3,4111769676,Prodej bytu 2+1 64 m²,"Ostrov, Karlovy Vary",Karlovarský kraj,50.307745,12.956295,3249000.0,50766,64,2+1,...,Osobní,C - Úsporná,NaN,231.0,1117.0,NaN,NaN,259.0,https://www.sreality.cz/detail/prodej/byt/2+1/...,2026-07-18T21:41:29.040753+00:00
4,3664875596,Prodej bytu 2+1 55 m²,"Chotěboř, Havlíčkův Brod",Kraj Vysočina,49.721356,15.671102,3390000.0,61636,55,2+1,...,Osobní,F - Velmi nehospodárná,NaN,36.0,1244.0,NaN,NaN,154.0,https://www.sreality.cz/detail/prodej/byt/2+1/...,2026-07-18T21:41:29.040753+00:00


## 2. Data Cleaning

In [2]:
# split colums
stavba_parts = df_raw['stavba'].str.split(',', expand=True)
loc_parts = df_raw['locality'].str.split(',', n=1, expand=True)

df_raw = df_raw.assign(
    # --- stavba ---
    construction = stavba_parts[0].str.strip(),
    condition    = stavba_parts[1].str.strip(),
    floor        = stavba_parts[2].str.extract(r'(\d+)')[0],

    # --- locality ---
    city     = loc_parts[0].str.split(' - ').str[0],
    district = loc_parts[0].str.split(' - ').str[1].fillna(loc_parts[1])
)

# Praha, use second part
df_raw.loc[df_raw['city'] == 'Praha', 'district'] = loc_parts[1]

# single-value locality, district = city
df_raw['district'] = df_raw['district'].fillna(df_raw['city'])


#clean topeni column. Priority order - Zdroj vytápění: take this if exists, else Topení:, else Otopná tělesa:
df_raw['heating'] = (
    df_raw['topeni']
    .str.extract(r'Zdroj vytápění:\s*([^;]+)')[0]
    .fillna(df_raw['topeni'].str.extract(r'Topení:\s*([^;]+)')[0])
    .fillna(df_raw['topeni'].str.extract(r'Otopná tělesa:\s*([^;]+)')[0])
    .str.strip()
)

df_raw = df_raw.drop(columns=[
    'stavba','locality','estate_id','title','url',
    'scraped_at','premise','seller', 'price_per_m2', 'scraped_at', 'studna', 'anuita','prislusenstvi','infrastruktura', 'topeni', 'is_new', 'telekomunikace',
])

In [3]:
cat = df_raw['category']

# extract as string
df_raw['rooms'] = cat.str.extract(r'(\d+)')[0].astype('string')
df_raw['kitchen'] = (
    cat.str.extract(r'\+(kk|1)')[0]
    .map({'kk': '0', '1': '1'})
    .astype('string')
)

# "a více" = kitchen = 1
df_raw.loc[cat.str.contains('a více', na=False), 'kitchen'] = '1'

# Atypický = both columns = "Atypický"
mask_atyp = cat.eq('Atypický')
df_raw.loc[mask_atyp, ['rooms', 'kitchen']] = 'Atypický'

# drop original column
df_raw = df_raw.drop(columns='category')

In [4]:
energy_map = {
    'A': 7,
    'B': 6,
    'C': 5,
    'D': 4,
    'E': 3,
    'F': 2,
    'G': 1
}

df_raw['energ_stitek'] = (
    df_raw['energ_stitek']
    .fillna('')
    .str.strip()
    .str.upper()
    .str.extract(r'^([A-G])')[0]
    .map(energy_map)
    .fillna(0)
    .astype(int)
)

df_raw['vlastnictvi'] = (
    df_raw['vlastnictvi']
    .map({
        'Osobní': 1,
        'Družstevní': 2
    })
    .fillna(0)
    .astype(int)
)

df_raw[['has_video', 'has_3d']] = df_raw[['has_video', 'has_3d']].astype(int)

In [5]:
condition_map = {
    'Špatný': 1,
    'Před rekonstrukcí': 2,
    'Projekt': 3,
    'V rekonstrukci': 4,
    'Ve výstavbě': 5,
    'Dobrý': 6,
    'Velmi dobrý': 7,
    'Po rekonstrukci': 8,
    'Novostavba': 9
}

df_raw['condition'] = (
    df_raw['condition']
    .map(condition_map)
    .fillna(0)
    .astype(int)
)

print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

Dataset shape: (17757, 22)


,region,latitude,longitude,price_czk,area_m2,has_video,has_3d,vlastnictvi,energ_stitek,dist_vecerka_m,...,dist_metro_m,dist_bus_mhd_m,construction,condition,floor,city,district,heating,rooms,kitchen
0,Jihočeský kraj,48.737864,14.102581,3290000.0,36,1,0,1,5,723.0,...,NaN,481.0,Cihlová,9,3,Černá v Pošumaví,Český Krumlov,Tepelné čerpadlo,1,0
1,Ústecký kraj,50.610939,13.636989,899000.0,25,0,0,2,1,28.0,...,NaN,68.0,Panelová,7,10,Litvínov,Horní Litvínov,Centrální dálkové,1,0
2,Jihočeský kraj,48.737864,14.102581,2690000.0,28,1,0,1,5,723.0,...,NaN,481.0,Cihlová,9,3,Černá v Pošumaví,Český Krumlov,Tepelné čerpadlo,1,0
3,Karlovarský kraj,50.307745,12.956295,3249000.0,64,0,0,1,5,231.0,...,NaN,259.0,Panelová,7,4,Ostrov,Karlovy Vary,Centrální dálkové,2,1
4,Kraj Vysočina,49.721356,15.671102,3390000.0,55,0,0,1,2,36.0,...,NaN,154.0,Smíšená,7,1,Chotěboř,Havlíčkův Brod,NaN,2,1


In [6]:
print('=== Missing Values ===')
missing = df_raw.isnull().sum()
print(missing[missing > 0])

print('\n=== Zero-Price Records ===')
zero_price = (df_raw['price_czk'] == 0).sum()
print(f'{zero_price} records have price = 0 ({zero_price/len(df_raw)*100:.1f}% of data)')
print('These are likely data entry errors and will be removed.')


=== Missing Values ===
dist_vecerka_m       48
dist_obchod_m      3065
dist_tram_m        9829
dist_metro_m      13044
dist_bus_mhd_m       16
construction         16
floor                16
heating            6764
dtype: int64

=== Zero-Price Records ===
0 records have price = 0 (0.0% of data)
These are likely data entry errors and will be removed.


In [7]:
df_raw.info()
df_raw.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17757 entries, 0 to 17756
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   region          17757 non-null  object 
 1   latitude        17757 non-null  float64
 2   longitude       17757 non-null  float64
 3   price_czk       17757 non-null  float64
 4   area_m2         17757 non-null  int64  
 5   has_video       17757 non-null  int64  
 6   has_3d          17757 non-null  int64  
 7   vlastnictvi     17757 non-null  int64  
 8   energ_stitek    17757 non-null  int64  
 9   dist_vecerka_m  17709 non-null  float64
 10  dist_obchod_m   14692 non-null  float64
 11  dist_tram_m     7928 non-null   float64
 12  dist_metro_m    4713 non-null   float64
 13  dist_bus_mhd_m  17741 non-null  float64
 14  construction    17741 non-null  object 
 15  condition       17757 non-null  int64  
 16  floor           17741 non-null  object 
 17  city            17757 non-null 

,latitude,longitude,price_czk,area_m2,has_video,has_3d,vlastnictvi,energ_stitek,dist_vecerka_m,dist_obchod_m,dist_tram_m,dist_metro_m,dist_bus_mhd_m,condition
count,17757.000000,17757.000000,1.775700e+04,17757.000000,17757.000000,17757.000000,17757.000000,17757.000000,17709.000000,14692.000000,7928.000000,4713.000000,17741.000000,17757.000000
mean,49.901594,15.169590,7.386380e+06,69.598693,0.226277,0.074393,1.096300,3.441741,419.141849,1136.158181,836.684788,1287.336304,191.653007,6.835896
std,0.492909,1.506708,5.483804e+06,55.758249,0.418432,0.262417,0.303851,2.461611,510.122030,950.853808,1090.079236,1182.832320,166.737971,1.702002
min,48.603779,12.174971,5.799000e+04,11.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,19.000000,1.000000,0.000000
25%,49.592800,14.332150,3.990000e+06,50.000000,0.000000,0.000000,1.000000,1.000000,141.000000,506.000000,165.750000,416.000000,97.000000,6.000000
50%,50.045903,14.512340,6.180000e+06,64.000000,0.000000,0.000000,1.000000,4.000000,263.000000,875.000000,345.000000,845.000000,157.000000,7.000000
75%,50.144407,16.498555,8.999000e+06,81.000000,0.000000,0.000000,1.000000,6.000000,507.000000,1423.000000,1066.000000,1746.000000,243.000000,8.000000
max,51.008696,18.766017,8.790000e+07,5989.000000,1.000000,1.000000,2.000000,7.000000,5902.000000,6382.000000,6402.000000,6211.000000,2659.000000,9.000000


## 3. Geographic Price Distribution

The color scale is clipped to the 5th-95th percentile of `price_czk` so a handful of luxury outliers; hovering over a point still shows its exact price.

In [8]:
df_map = df_raw[
    (df_raw['price_czk'] > 0) &
    df_raw['latitude'].notna() &
    df_raw['longitude'].notna()
].copy()

p_low, p_high = df_map['price_czk'].quantile([0.05, 0.95])

fig = px.scatter_map(
    df_map,
    lat='latitude',
    lon='longitude',
    color='price_czk',
    color_continuous_scale='Viridis',
    range_color=[p_low, p_high],
    zoom=6,
    center={'lat': 49.8, 'lon': 15.5},
    map_style='carto-positron',
    opacity=0.6,
    hover_data={
        'city': True,
        'district': True,
        'price_czk': ':,.0f',
        'latitude': False,
        'longitude': False,
    },
    labels={'price_czk': 'Price (CZK)'},
)

fig.update_traces(marker=dict(size=7))
fig.update_layout(
    width=1300,
    height=950,
    margin=dict(l=0, r=0, t=50, b=0),
    title=dict(
        text='Where House Prices Run High and Low Across Czechia',
        x=0.02,
        font=dict(size=18, family='Helvetica, Arial, sans-serif', color='#333'),
    ),
    coloraxis_colorbar=dict(title='Price (CZK)', tickformat=',.0f', len=0.75),
    font=dict(family='Helvetica, Arial, sans-serif', size=12),
    paper_bgcolor='white',
)

fig.show()
